In [1]:
from dotenv import load_dotenv
from IPython.display import Markdown
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

In [2]:
load_dotenv()

True

#### Chat Model & Template

In [3]:
chat = ChatGoogleGenerativeAI(
    model = 'gemini-3.1-flash-lite-preview',
    temperature = 0,
    max_output_tokens = 200,
    seed = 0
)

In [4]:
chat_template_skills = ChatPromptTemplate.from_template('''
Give the 5 of most important tools for a {profession}.
Answer by only listing the names.
''')

chat_template_projects = ChatPromptTemplate.from_template('''
Can you suggest 3 'must-do' intermediate level projects for a {profession}.
Answer by only listing the names.
''')

#### Parallel Chains

In [5]:
from langchain_core.runnables import RunnableParallel
from langchain_core.output_parsers import StrOutputParser

In [6]:
str_parser = StrOutputParser()

##### Chain 1

In [7]:
chain_skills = chat_template_skills | chat | str_parser

In [8]:
%%time
chain_skills.invoke({'profession': 'AI Engineer'})

CPU times: total: 78.1 ms
Wall time: 1.7 s


'1. Python\n2. PyTorch\n3. TensorFlow\n4. Hugging Face Transformers\n5. Docker'

##### Chain 2

In [9]:
chain_projects = chat_template_projects | chat | str_parser

In [10]:
%%time
chain_projects.invoke({'profession': 'AI Engineer'})

CPU times: total: 15.6 ms
Wall time: 840 ms


'1. RAG-based Question Answering System\n2. Fine-tuned LLM for Domain-Specific Tasks\n3. End-to-End MLOps Pipeline with Model Monitoring'

##### Compose as Parallel

In [11]:
chain_parallel = RunnableParallel({'skills': chain_skills, 'projects': chain_projects}) # key can be anything

In [12]:
%%time
chain_parallel.invoke({'profession': 'AI Engineer'})

CPU times: total: 31.2 ms
Wall time: 1.45 s


{'skills': '1. Python\n2. PyTorch\n3. TensorFlow\n4. Hugging Face Transformers\n5. Docker',
 'projects': '1. RAG-based Question Answering System\n2. Fine-tuned LLM for Domain-Specific Tasks\n3. End-to-End MLOps Pipeline with Model Monitoring'}

> invoking runnables in parrallel is more time efficient

> its because the sum of wall times of individual invokes is more than that of parallel invoke

##### Visualize Chain

In [13]:
chain_parallel.get_graph().print_ascii()

                  +--------------------------------+                     
                  | Parallel<skills,projects>Input |                     
                  +--------------------------------+                     
                       ****                  ****                        
                   ****                          ****                    
                 **                                  **                  
  +--------------------+                       +--------------------+    
  | ChatPromptTemplate |                       | ChatPromptTemplate |    
  +--------------------+                       +--------------------+    
             *                                            *              
             *                                            *              
             *                                            *              
+------------------------+                   +------------------------+  
| ChatGoogleGenerativeAI |            

> batch() invokes the same runnable with different input values

> RunnableParallel invokes several runnables with same input values

#### Complex Chain

In [14]:
chat_template_time = ChatPromptTemplate.from_template('''
I am an intermediate level programmar.
Tell me how much time it should take me to learn {skills} and complete {projects}.
Answer by giving the estimated time for each and no extra wording.
''')

In [15]:
chain_time = (
    {'skills': chain_skills, 'projects': chain_projects} | # using {} is shorthand for RunnableParallel in LCEL
    chat_template_time |
    chat |
    str_parser
)

In [16]:
display(Markdown(chain_time.invoke({'profession': 'AI Engineer'})))

1. **Python:** 2–4 weeks (assuming proficiency in another language).
2. **PyTorch:** 3–5 weeks.
3. **TensorFlow:** 3–5 weeks.
4. **Hugging Face Transformers:** 2–3 weeks.
5. **Docker:** 1–2 weeks.

1. **RAG-based Question Answering System:** 2–3 weeks.
2. **Fine-tuned LLM for Domain-Specific Tasks:** 3–4 weeks.
3. **End-to-End MLOps Pipeline with Model Monitoring:** 4–6 weeks.

In [17]:
chain_time.get_graph().print_ascii()

                  +--------------------------------+                     
                  | Parallel<skills,projects>Input |                     
                  +--------------------------------+                     
                       ****                  ****                        
                   ****                          ****                    
                 **                                  **                  
  +--------------------+                       +--------------------+    
  | ChatPromptTemplate |                       | ChatPromptTemplate |    
  +--------------------+                       +--------------------+    
             *                                            *              
             *                                            *              
             *                                            *              
+------------------------+                   +------------------------+  
| ChatGoogleGenerativeAI |            